In [ ]:



from __future__ import annotations

import json
import random
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.rdchem import RWMol


FLOAT_TOL = 1.0e-4


@dataclass
class PolymerBuildResult:
    

    copolymer_name: str
    sequence: List[str]
    cyclic: bool
    expected_charge: float
    actual_charge: float
    node_list: List[str]
    xyz_elements: List[str]
    charge_list: List[float]
    xyz_path: Path
    chg_path: Path
    node_list_path: Path


BOND_LIST = [
    Chem.rdchem.BondType.UNSPECIFIED,
    Chem.rdchem.BondType.SINGLE,
    Chem.rdchem.BondType.DOUBLE,
    Chem.rdchem.BondType.TRIPLE,
    Chem.rdchem.BondType.QUADRUPLE,
    Chem.rdchem.BondType.QUINTUPLE,
    Chem.rdchem.BondType.HEXTUPLE,
    Chem.rdchem.BondType.ONEANDAHALF,
    Chem.rdchem.BondType.TWOANDAHALF,
    Chem.rdchem.BondType.THREEANDAHALF,
    Chem.rdchem.BondType.FOURANDAHALF,
    Chem.rdchem.BondType.FIVEANDAHALF,
    Chem.rdchem.BondType.AROMATIC,
    Chem.rdchem.BondType.IONIC,
    Chem.rdchem.BondType.HYDROGEN,
    Chem.rdchem.BondType.THREECENTER,
    Chem.rdchem.BondType.DATIVEONE,
    Chem.rdchem.BondType.DATIVE,
    Chem.rdchem.BondType.DATIVEL,
    Chem.rdchem.BondType.DATIVER,
    Chem.rdchem.BondType.OTHER,
    Chem.rdchem.BondType.ZERO,
]


def safe_name(name: str) -> str:
    
    return "".join(ch if ch.isalnum() or ch in "._+-" else "_" for ch in str(name))


def load_mapped_charges(path: Path) -> Dict[str, Any]:
    
    with Path(path).open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if "mapped_charges" not in payload:
        raise ValueError("Public message removed for release.")
    return payload


def build_sequence(repeating_unit: Dict[str, int], random_seed: int) -> List[str]:
    
    counts = {name: int(count) for name, count in repeating_unit.items()}
    if any(count < 0 for count in counts.values()) or sum(counts.values()) <= 0:
        raise ValueError("Public message removed for release.")

    random.seed(int(random_seed))
    names = list(counts.keys())
    tracker = {name: 0 for name in names}
    sequence: List[str] = []
    for _ in range(sum(counts.values())):
        while True:
            name = random.choice(names)
            if tracker[name] < counts[name]:
                tracker[name] += 1
                sequence.append(name)
                break
    return sequence


def build_block_sequence(repeating_unit: Dict[str, int], number_of_blocks: int) -> List[str]:
    
    n_blocks = int(number_of_blocks)
    if n_blocks <= 0:
        raise ValueError("Public message removed for release.")
    counts = {name: int(count) for name, count in repeating_unit.items()}
    if any(count < 0 for count in counts.values()) or sum(counts.values()) <= 0:
        raise ValueError("Public message removed for release.")

    sequence: List[str] = []
    for _ in range(n_blocks):
        for name, repeats in counts.items():
            sequence.extend([name] * repeats)
    return sequence


def build_homopolymer_sequence(name: str, repeating_unit: int) -> List[str]:
    
    n_repeat = int(repeating_unit)
    if n_repeat <= 0:
        raise ValueError("Public message removed for release.")
    return [str(name)] * n_repeat


def node_label(atom: Chem.Atom, unit_name: str, repeat_index1: int) -> str:
    
    symbol = atom.GetSymbol()
    idx = atom.GetIdx()
    prefix = "*" if symbol == "*" else symbol
    return f"{prefix}{idx}_{unit_name}_{repeat_index1}"


def base_node_label(atom: Chem.Atom, unit_name: str) -> str:
    
    symbol = atom.GetSymbol()
    idx = atom.GetIdx()
    prefix = "*" if symbol == "*" else symbol
    return f"{prefix}{idx}_{unit_name}"


def node_to_element(node: str) -> str:
    
    if node.startswith("*"):
        return "*"
    match = re.match(r"([A-Z][a-z]*)\d+_", node)
    if not match:
        raise ValueError("Public message removed for release.")
    return match.group(1)


def capped_node_element(node: str) -> str:
    
    return "H" if node.startswith("*") else node_to_element(node)


def formal_charge_for_node(node: str, mapped_payloads: Dict[str, Dict[str, Any]]) -> int:
    
    parts = node.rsplit("_", 1)
    if len(parts) != 2:
        return 0
    base = parts[0]
    unit_name = base.rsplit("_", 1)[-1]
    payload = mapped_payloads.get(unit_name)
    if payload is None:
        return 0
    entry = payload["mapped_charges"].get(base)
    if entry is None:
        return 0
    return int(entry.get("formal_charge", 0))


def collect_polymer_graph(
    smiles_by_name: Dict[str, str],
    mapped_payloads: Dict[str, Dict[str, Any]],
    sequence: Sequence[str],
    cyclic: bool = False,
) -> Tuple[Dict[str, float], Dict[str, int], List[Tuple[str, str]], List[Tuple[Tuple[str, str], int]]]:
    
    all_charges: Dict[str, float] = {}
    formal_charges: Dict[str, int] = {}
    all_bonds: List[Tuple[str, str]] = []
    all_bonds_with_order: List[Tuple[Tuple[str, str], int]] = []
    dummy_bonds: List[Tuple[str, str]] = []

    for repeat_idx0, unit_name in enumerate(sequence):
        repeat_idx1 = repeat_idx0 + 1
        mol = Chem.MolFromSmiles(smiles_by_name[unit_name])
        if mol is None:
            raise ValueError("Public message removed for release.")
        mol_with_h = Chem.AddHs(mol)
        mapped = mapped_payloads[unit_name]["mapped_charges"]

        unit_dummy_bonds: List[Tuple[str, str]] = []
        for atom in mol_with_h.GetAtoms():
            base_label = base_node_label(atom, unit_name)
            if base_label not in mapped:
                raise KeyError("Public message removed for release.")
            full_label = node_label(atom, unit_name, repeat_idx1)
            all_charges[full_label] = float(mapped[base_label]["charge"])
            q = int(mapped[base_label].get("formal_charge", atom.GetFormalCharge()))
            if q != 0:
                formal_charges[full_label] = q

        for bond in mol_with_h.GetBonds():
            begin = mol_with_h.GetAtomWithIdx(bond.GetBeginAtomIdx())
            end = mol_with_h.GetAtomWithIdx(bond.GetEndAtomIdx())
            begin_node = node_label(begin, unit_name, repeat_idx1)
            end_node = node_label(end, unit_name, repeat_idx1)
            order = BOND_LIST.index(bond.GetBondType())
            all_bonds.append((begin_node, end_node))
            all_bonds_with_order.append(((begin_node, end_node), order))
            if begin.GetSymbol() == "*" or end.GetSymbol() == "*":
                unit_dummy_bonds.append((begin_node, end_node))

        if len(unit_dummy_bonds) != 2:
            raise ValueError("Public message removed for release.")
        dummy_bonds.extend(unit_dummy_bonds)

    non_virtual_atoms: List[str] = []
    for bond in dummy_bonds:
        left, right = bond
        if left.startswith("*"):
            non_virtual_atoms.append(right)
        else:
            non_virtual_atoms.append(left)

    inter_bonds: List[Tuple[str, str]] = []
    for idx in range(1, len(non_virtual_atoms) - 1, 2):
        inter_bonds.append((non_virtual_atoms[idx], non_virtual_atoms[idx + 1]))

    if cyclic:
        
        inter_bonds.append((non_virtual_atoms[0], non_virtual_atoms[-1]))
        final_charges = {
            key: value
            for key, value in all_charges.items()
            if not key.startswith("*")
        }
    else:
        
        end_dummy_charge_dict: Dict[str, float] = {}
        first_dummy_key: str | None = None
        last_dummy_key: str | None = None
        for atom_name in all_charges:
            if atom_name.startswith("*"):
                if first_dummy_key is None:
                    first_dummy_key = atom_name
                last_dummy_key = atom_name
        if first_dummy_key is None or last_dummy_key is None:
            raise ValueError("Public message removed for release.")
        end_dummy_charge_dict[first_dummy_key] = all_charges[first_dummy_key]
        end_dummy_charge_dict[last_dummy_key] = all_charges[last_dummy_key]

        final_charges = {
            key: value
            for key, value in all_charges.items()
            if not key.startswith("*") or key in end_dummy_charge_dict
        }
        inter_bonds.insert(0, dummy_bonds[0])
        inter_bonds.append(dummy_bonds[-1])

    inter_bonds_with_order = [(bond, 1) for bond in inter_bonds]

    intra_bonds = [bond for bond in all_bonds if not bond[0].startswith("*") and not bond[1].startswith("*")]
    intra_bonds_with_order = [
        item
        for item in all_bonds_with_order
        if not item[0][0].startswith("*") and not item[0][1].startswith("*")
    ]

    final_bonds = intra_bonds + inter_bonds
    final_bonds_with_order = intra_bonds_with_order + inter_bonds_with_order
    return final_charges, formal_charges, final_bonds, final_bonds_with_order


def graph_to_mol(
    node_list: Sequence[str],
    bonds_with_order: Sequence[Tuple[Tuple[str, str], int]],
    formal_charges: Dict[str, int],
) -> Chem.Mol:
    
    rw_mol = Chem.RWMol()
    node_to_idx: Dict[str, int] = {}
    for node in node_list:
        atom = Chem.Atom(node_to_element(node))
        atom_idx = rw_mol.AddAtom(atom)
        node_to_idx[node] = atom_idx

    for (node1, node2), order_idx in bonds_with_order:
        if node1 not in node_to_idx or node2 not in node_to_idx:
            raise KeyError("Public message removed for release.")
        rw_mol.AddBond(node_to_idx[node1], node_to_idx[node2], BOND_LIST[int(order_idx)])

    for node, charge in formal_charges.items():
        if node in node_to_idx:
            rw_mol.GetAtomWithIdx(node_to_idx[node]).SetFormalCharge(int(charge))

    mol = rw_mol.GetMol()
    Chem.SanitizeMol(mol)
    return mol


def cap_terminal_dummy_atoms(mol: Chem.Mol) -> Chem.Mol:
    
    dummy_indices = [atom.GetIdx() for atom in mol.GetAtoms() if atom.GetSymbol() == "*"]
    if len(dummy_indices) != 2:
        raise ValueError("Public message removed for release.")
    rw_mol = RWMol(mol)
    for idx in dummy_indices:
        rw_mol.ReplaceAtom(idx, Chem.Atom(1))
        rw_mol.GetAtomWithIdx(idx).SetNoImplicit(False)
    rw_mol.UpdatePropertyCache(strict=False)
    capped = rw_mol.GetMol()
    Chem.SanitizeMol(capped)
    return capped


def _write_openbabel_structure_files(mol: Chem.Mol, prefixes: Sequence[Path]) -> None:
    
    try:
        from openbabel import openbabel as ob  
    except Exception as exc:
        print("Public message removed for release.")
        return

    mol_block = Chem.MolToMolBlock(mol)
    for prefix in prefixes:
        prefix = Path(prefix)
        prefix.parent.mkdir(parents=True, exist_ok=True)
        ob_conversion = ob.OBConversion()
        ob_conversion.SetInAndOutFormats("sdf", "mol2")
        ob_mol = ob.OBMol()
        ok = ob_conversion.ReadString(ob_mol, mol_block)
        if not ok:
            print("Public message removed for release.")
            continue
        ob_conversion.WriteFile(ob_mol, str(prefix.with_suffix(".mol2")))
        ob_conversion.SetOutFormat("pdb")
        ob_conversion.WriteFile(ob_mol, str(prefix.with_suffix(".pdb")))


def write_xyz(mol: Chem.Mol, xyz_path: Path, extra_structure_prefixes: Sequence[Path] | None = None) -> List[str]:
    
    work_mol = Chem.Mol(mol)
    status = AllChem.EmbedMolecule(work_mol, useRandomCoords=True, randomSeed=20260428)
    if status != 0:
        raise RuntimeError("Public message removed for release.")
    try:
        AllChem.MMFFOptimizeMolecule(work_mol, mmffVariant="MMFF94")
    except Exception:
        AllChem.UFFOptimizeMolecule(work_mol)

    xyz_path.parent.mkdir(parents=True, exist_ok=True)
    xyz_text = Chem.MolToXYZBlock(work_mol)
    xyz_path.write_text(xyz_text, encoding="utf-8")
    if extra_structure_prefixes:
        for prefix in extra_structure_prefixes:
            prefix = Path(prefix)
            prefix.parent.mkdir(parents=True, exist_ok=True)
            prefix.with_suffix(".xyz").write_text(xyz_text, encoding="utf-8")
        _write_openbabel_structure_files(work_mol, extra_structure_prefixes)
    elements = [line.split()[0] for line in xyz_text.splitlines()[2:] if line.split()]
    return elements


def write_chg(
    xyz_path: Path,
    charges: Sequence[float],
    chg_path: Path,
    extra_chg_paths: Sequence[Path] | None = None,
) -> None:
    
    lines = xyz_path.read_text(encoding="utf-8").splitlines()
    coord_lines = [line for line in lines[2:] if line.split()]
    if len(coord_lines) != len(charges):
        raise ValueError("Public message removed for release.")

    chg_path.parent.mkdir(parents=True, exist_ok=True)
    with chg_path.open("w", encoding="utf-8", newline="\n") as handle:
        for line, charge in zip(coord_lines, charges):
            parts = line.split()
            handle.write(
                f"{parts[0]} {float(parts[1]): .6f} {float(parts[2]): .6f} "
                f"{float(parts[3]): .6f} {float(charge): .10f}\n"
            )
    if extra_chg_paths:
        for extra_path in extra_chg_paths:
            extra_path = Path(extra_path)
            extra_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(chg_path, extra_path)


def build_random_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
    random_seed: int = 2,
) -> PolymerBuildResult:
    
    base_dir = Path(base_dir)
    sequence = build_sequence(repeating_unit, random_seed)
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=False,
        repeating_unit_for_expected_charge=repeating_unit,
    )


def build_polymer_from_sequence(
    smiles_by_name: Dict[str, str],
    sequence: Sequence[str],
    mapped_charge_paths: Dict[str, Path],
    polymer_name: str,
    base_dir: Path,
    cyclic: bool = False,
    repeating_unit_for_expected_charge: Dict[str, int] | None = None,
) -> PolymerBuildResult:
    
    if not sequence:
        raise ValueError("Public message removed for release.")

    base_dir = Path(base_dir)
    mapped_payloads = {
        name: load_mapped_charges(Path(path))
        for name, path in mapped_charge_paths.items()
    }
    missing = sorted(set(sequence).difference(smiles_by_name))
    if missing:
        raise KeyError("Public message removed for release.")
    missing_mapped = sorted(set(sequence).difference(mapped_payloads))
    if missing_mapped:
        raise KeyError("Public message removed for release.")

    if repeating_unit_for_expected_charge is None:
        repeating_unit_for_expected_charge = {
            name: int(sequence.count(name))
            for name in dict.fromkeys(sequence)
        }

    charge_dict, formal_charges, _, bonds_with_order = collect_polymer_graph(
        smiles_by_name=smiles_by_name,
        mapped_payloads=mapped_payloads,
        sequence=sequence,
        cyclic=bool(cyclic),
    )

    expected_charge = sum(
        int(repeating_unit_for_expected_charge[name]) * float(mapped_payloads[name]["formal_charge"])
        for name in repeating_unit_for_expected_charge
    )
    if not cyclic:
        
        
        
        
        
        terminal_dummy_nodes = [node for node in charge_dict if node.startswith("*")]
        if len(terminal_dummy_nodes) != 2:
            raise ValueError("Public message removed for release.")
        current_charge = float(np.sum([float(value) for value in charge_dict.values()]))
        terminal_delta = expected_charge - current_charge
        if abs(terminal_delta) > FLOAT_TOL:
            correction = terminal_delta / 2.0
            for node in terminal_dummy_nodes:
                charge_dict[node] = float(charge_dict[node]) + correction

    node_list = list(charge_dict.keys())
    charge_list = [float(charge_dict[node]) for node in node_list]
    expected_elements = [capped_node_element(node) for node in node_list]

    mol = graph_to_mol(node_list, bonds_with_order, formal_charges)
    capped_mol = mol if cyclic else cap_terminal_dummy_atoms(mol)

    safe_polymer = safe_name(polymer_name)
    xyz_path = base_dir / "polymer_structure" / f"{safe_polymer}.xyz"
    chg_path = base_dir / "polymer_topology" / f"{safe_polymer}.chg"
    node_list_path = base_dir / "polymer_node_list.json"

    legacy_prefixes = [
        base_dir / safe_polymer,
        base_dir / f"polymer_{safe_polymer}",
        base_dir / "polymer_structure" / safe_polymer,
        base_dir / "polymer_structure" / f"polymer_{safe_polymer}",
    ]
    xyz_elements = write_xyz(capped_mol, xyz_path, extra_structure_prefixes=legacy_prefixes)
    if xyz_elements != expected_elements:
        mismatch = [
            (idx + 1, got, expected)
            for idx, (got, expected) in enumerate(zip(xyz_elements, expected_elements))
            if got != expected
        ][:10]
        raise ValueError("Public message removed for release.")

    actual_charge = float(np.sum(charge_list))
    if abs(actual_charge - expected_charge) > FLOAT_TOL:
        raise ValueError(
            "Public message removed for release."
        )

    node_list_path.write_text(json.dumps(node_list, ensure_ascii=False, indent=2), encoding="utf-8")
    write_chg(
        xyz_path,
        charge_list,
        chg_path,
        extra_chg_paths=[
            base_dir / f"{safe_polymer}.chg",
            base_dir / f"polymer_{safe_polymer}.chg",
            base_dir / "polymer_topology" / f"polymer_{safe_polymer}.chg",
        ],
    )

    return PolymerBuildResult(
        copolymer_name=polymer_name,
        cyclic=bool(cyclic),
        sequence=list(sequence),
        expected_charge=expected_charge,
        actual_charge=actual_charge,
        node_list=node_list,
        xyz_elements=xyz_elements,
        charge_list=charge_list,
        xyz_path=xyz_path,
        chg_path=chg_path,
        node_list_path=node_list_path,
    )


def build_block_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    number_of_blocks: int,
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    
    sequence = build_block_sequence(repeating_unit, number_of_blocks)
    expanded_counts = {name: int(count) * int(number_of_blocks) for name, count in repeating_unit.items()}
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=False,
        repeating_unit_for_expected_charge=expanded_counts,
    )


def build_cyclic_block_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    number_of_blocks: int,
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    
    sequence = build_block_sequence(repeating_unit, number_of_blocks)
    expanded_counts = {name: int(count) * int(number_of_blocks) for name, count in repeating_unit.items()}
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=True,
        repeating_unit_for_expected_charge=expanded_counts,
    )


def build_cyclic_random_copolymer(
    smiles_by_name: Dict[str, str],
    repeating_unit: Dict[str, int],
    mapped_charge_paths: Dict[str, Path],
    copolymer_name: str,
    base_dir: Path,
    random_seed: int = 2,
) -> PolymerBuildResult:
    
    sequence = build_sequence(repeating_unit, random_seed)
    return build_polymer_from_sequence(
        smiles_by_name=smiles_by_name,
        sequence=sequence,
        mapped_charge_paths=mapped_charge_paths,
        polymer_name=copolymer_name,
        base_dir=base_dir,
        cyclic=True,
        repeating_unit_for_expected_charge=repeating_unit,
    )


def build_homopolymer(
    smiles: str,
    repeating_unit: int,
    mapped_charge_path: Path,
    polymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    
    sequence = build_homopolymer_sequence(polymer_name, repeating_unit)
    return build_polymer_from_sequence(
        smiles_by_name={polymer_name: smiles},
        sequence=sequence,
        mapped_charge_paths={polymer_name: Path(mapped_charge_path)},
        polymer_name=polymer_name,
        base_dir=base_dir,
        cyclic=False,
        repeating_unit_for_expected_charge={polymer_name: int(repeating_unit)},
    )


def build_cyclic_homopolymer(
    smiles: str,
    repeating_unit: int,
    mapped_charge_path: Path,
    polymer_name: str,
    base_dir: Path,
) -> PolymerBuildResult:
    
    sequence = build_homopolymer_sequence(polymer_name, repeating_unit)
    return build_polymer_from_sequence(
        smiles_by_name={polymer_name: smiles},
        sequence=sequence,
        mapped_charge_paths={polymer_name: Path(mapped_charge_path)},
        polymer_name=polymer_name,
        base_dir=base_dir,
        cyclic=True,
        repeating_unit_for_expected_charge={polymer_name: int(repeating_unit)},
    )


In [ ]:



from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Dict, List


FLOAT_TOL = 1.0e-4


def read_chg(chg_path: Path) -> List[Dict[str, Any]]:
    
    rows: List[Dict[str, Any]] = []
    with Path(chg_path).open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, 1):
            parts = line.split()
            if not parts:
                continue
            if len(parts) < 5:
                raise ValueError("Public message removed for release.")
            rows.append(
                {
                    "index1": len(rows) + 1,
                    "element": parts[0],
                    "charge": float(parts[-1]),
                }
            )
    return rows


def read_xyz_elements(xyz_path: Path) -> List[str]:
    
    lines = Path(xyz_path).read_text(encoding="utf-8").splitlines()
    return [line.split()[0] for line in lines[2:] if line.split()]


def load_json(path: Path) -> Any:
    
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def node_to_final_element(node: str) -> str:
    
    if node.startswith("*"):
        return "H"
    
    chars = []
    for ch in node:
        if ch.isdigit() or ch == "_":
            break
        chars.append(ch)
    if not chars:
        raise ValueError("Public message removed for release.")
    return "".join(chars)


def validate_repeat_unit_payload(name: str, payload: Dict[str, Any]) -> Dict[str, Any]:
    
    mapped = payload["mapped_charges"]
    formal_charge = float(payload["formal_charge"])
    non_dummy_sum = 0.0
    dummy_nodes = []
    for node, entry in mapped.items():
        charge = float(entry["charge"])
        if entry["is_dummy_replacement"]:
            dummy_nodes.append(
                {
                    "node": node,
                    "gaussian_index1": int(entry["gaussian_index1"]),
                    "gaussian_symbol": entry["gaussian_symbol"],
                    "charge": charge,
                }
            )
            if entry["gaussian_symbol"] != "H":
                raise ValueError("Public message removed for release.")
        else:
            non_dummy_sum += charge

    if len(dummy_nodes) != 2:
        raise ValueError("Public message removed for release.")
    if abs(non_dummy_sum - formal_charge) > FLOAT_TOL:
        raise ValueError(
            "Public message removed for release."
        )
    return {
        "name": name,
        "formal_charge": formal_charge,
        "non_dummy_charge_sum": non_dummy_sum,
        "dummy_nodes": dummy_nodes,
    }


def validate_full_result(
    base_dir: Path,
    repeating_unit: Dict[str, int],
    mapped_charge_paths: Dict[str, Path],
    node_list_path: Path,
    xyz_path: Path,
    chg_path: Path,
    report_path: Path | None = None,
    old_reference_charge: float | None = 4.4005007164,
) -> Dict[str, Any]:
    
    base_dir = Path(base_dir)
    unit_reports = {}
    expected_charge = 0.0
    for name, count in repeating_unit.items():
        payload = load_json(Path(mapped_charge_paths[name]))
        unit_reports[name] = validate_repeat_unit_payload(name, payload)
        expected_charge += int(count) * float(payload["formal_charge"])

    node_list = load_json(Path(node_list_path))
    xyz_elements = read_xyz_elements(Path(xyz_path))
    chg_rows = read_chg(Path(chg_path))
    chg_elements = [row["element"] for row in chg_rows]
    node_elements = [node_to_final_element(node) for node in node_list]

    if len(node_list) != len(xyz_elements):
        raise ValueError("Public message removed for release.")
    if len(chg_rows) != len(xyz_elements):
        raise ValueError("Public message removed for release.")
    if node_elements != xyz_elements:
        raise ValueError("Public message removed for release.")
    if chg_elements != xyz_elements:
        raise ValueError("Public message removed for release.")

    actual_charge = sum(float(row["charge"]) for row in chg_rows)
    if abs(actual_charge - expected_charge) > FLOAT_TOL:
        raise ValueError(
            "Public message removed for release."
        )

    report = {
        "base_dir": str(base_dir.resolve()),
        "status": "pass",
        "expected_polymer_charge": expected_charge,
        "actual_polymer_charge": actual_charge,
        "old_reference_charge": old_reference_charge,
        "atom_count": len(xyz_elements),
        "unit_reports": unit_reports,
        "files": {
            "node_list": str(Path(node_list_path)),
            "xyz": str(Path(xyz_path)),
            "chg": str(Path(chg_path)),
        },
    }
    if report_path is not None:
        Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report



In [ ]:

EXCEL_FILE = "System_block_copolymer.xlsx"
POLYMER_MODE = "block"  
RANDOM_SEED = 2

import json
from pathlib import Path
from typing import Dict

import pandas as pd


def read_polymer_rows(excel_path: Path) -> pd.DataFrame:
    
    df = pd.read_excel(excel_path)
    missing = {"Name", "SMILES", "repeating unit"}.difference(df.columns)
    if missing:
        raise ValueError("Public message removed for release.")

    work_df = df.dropna(subset=["Name", "SMILES", "repeating unit"]).copy()
    work_df["Name"] = work_df["Name"].astype(str).str.strip()
    work_df["SMILES"] = work_df["SMILES"].astype(str).str.strip()
    work_df = work_df[
        (work_df["Name"] != "")
        & (work_df["SMILES"] != "")
        & (work_df["Name"].str.lower() != "nan")
        & (work_df["SMILES"].str.lower() != "nan")
    ].copy()

    if "is polymer" in work_df.columns:
        polymer_df = work_df[work_df["is polymer"].astype(bool)].copy()
    else:
        polymer_df = work_df.copy()

    if polymer_df.empty:
        raise ValueError("Public message removed for release.")
    if polymer_df["Name"].astype(str).duplicated().any():
        duplicated = polymer_df.loc[polymer_df["Name"].astype(str).duplicated(), "Name"].astype(str).tolist()
        raise ValueError("Public message removed for release.")
    return polymer_df


def get_copolymer_name(polymer_df: pd.DataFrame, default_name: str | None = None) -> str:
    
    if "copolymer_name" in polymer_df.columns:
        value = polymer_df["copolymer_name"].iloc[0]
        if not pd.isna(value) and str(value).strip() and str(value).lower() != "nan":
            return str(value)
    if default_name is not None:
        return default_name
    raise ValueError("Public message removed for release.")


def build_input_dicts(polymer_df: pd.DataFrame):
    
    smiles_by_name = {str(row["Name"]): str(row["SMILES"]) for _, row in polymer_df.iterrows()}
    repeating_unit = {str(row["Name"]): int(row["repeating unit"]) for _, row in polymer_df.iterrows()}
    mapped_charge_paths = {name: Path("mapped_charges") / f"{safe_name(name)}_mapped_charges.json" for name in smiles_by_name}
    missing = [str(path) for path in mapped_charge_paths.values() if not Path(path).exists()]
    if missing:
        raise FileNotFoundError("Public message removed for release." + ", ".join(missing))
    return smiles_by_name, repeating_unit, mapped_charge_paths


def save_repeat_unit_atom_lists(mapped_charge_paths: Dict[str, Path]) -> None:
    
    for name, path in mapped_charge_paths.items():
        payload = load_mapped_charges(Path(path))
        atom_list = list(payload["mapped_charges"].keys())
        out_path = Path(f"{safe_name(name)}_atom_list.json")
        out_path.write_text(json.dumps(atom_list, ensure_ascii=False, indent=2), encoding="utf-8")
        print("Public message removed for release.")


base_dir = Path.cwd()
polymer_df = read_polymer_rows(Path(EXCEL_FILE))
smiles_by_name, repeating_unit, mapped_charge_paths = build_input_dicts(polymer_df)
save_repeat_unit_atom_lists(mapped_charge_paths)

if POLYMER_MODE == "homopolymer":
    reports = []
    for _, row in polymer_df.iterrows():
        name = str(row["Name"])
        result = build_homopolymer(
            smiles=smiles_by_name[name],
            repeating_unit=repeating_unit[name],
            mapped_charge_path=mapped_charge_paths[name],
            polymer_name=name,
            base_dir=base_dir,
        )
        report = validate_full_result(
            base_dir=base_dir,
            repeating_unit={name: repeating_unit[name]},
            mapped_charge_paths={name: mapped_charge_paths[name]},
            node_list_path=result.node_list_path,
            xyz_path=result.xyz_path,
            chg_path=result.chg_path,
            report_path=base_dir / f"validation_report_{safe_name(name)}.json",
            old_reference_charge=None,
        )
        reports.append(report)
    print(json.dumps(reports, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "cyclic-homopolymer":
    reports = []
    for _, row in polymer_df.iterrows():
        name = str(row["Name"])
        result = build_cyclic_homopolymer(
            smiles=smiles_by_name[name],
            repeating_unit=repeating_unit[name],
            mapped_charge_path=mapped_charge_paths[name],
            polymer_name=name,
            base_dir=base_dir,
        )
        report = validate_full_result(
            base_dir=base_dir,
            repeating_unit={name: repeating_unit[name]},
            mapped_charge_paths={name: mapped_charge_paths[name]},
            node_list_path=result.node_list_path,
            xyz_path=result.xyz_path,
            chg_path=result.chg_path,
            report_path=base_dir / f"validation_report_{safe_name(name)}.json",
            old_reference_charge=None,
        )
        reports.append(report)
    print(json.dumps(reports, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "random":
    copolymer_name = get_copolymer_name(polymer_df)
    result = build_random_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
        random_seed=RANDOM_SEED,
    )
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "cyclic-random":
    copolymer_name = get_copolymer_name(polymer_df)
    result = build_cyclic_random_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
        random_seed=RANDOM_SEED,
    )
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=repeating_unit,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "block":
    copolymer_name = get_copolymer_name(polymer_df)
    if "Number of blocks" not in polymer_df.columns:
        raise ValueError("Public message removed for release.")
    number_of_blocks = int(polymer_df["Number of blocks"].iloc[0])
    result = build_block_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        number_of_blocks=number_of_blocks,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
    )
    validation_counts = {name: count * number_of_blocks for name, count in repeating_unit.items()}
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=validation_counts,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

elif POLYMER_MODE == "cyclic-block":
    copolymer_name = get_copolymer_name(polymer_df)
    if "Number of blocks" not in polymer_df.columns:
        raise ValueError("Public message removed for release.")
    number_of_blocks = int(polymer_df["Number of blocks"].iloc[0])
    result = build_cyclic_block_copolymer(
        smiles_by_name=smiles_by_name,
        repeating_unit=repeating_unit,
        number_of_blocks=number_of_blocks,
        mapped_charge_paths=mapped_charge_paths,
        copolymer_name=copolymer_name,
        base_dir=base_dir,
    )
    validation_counts = {name: count * number_of_blocks for name, count in repeating_unit.items()}
    report = validate_full_result(
        base_dir=base_dir,
        repeating_unit=validation_counts,
        mapped_charge_paths=mapped_charge_paths,
        node_list_path=result.node_list_path,
        xyz_path=result.xyz_path,
        chg_path=result.chg_path,
        report_path=base_dir / "validation_report.json",
        old_reference_charge=None,
    )
    print(json.dumps(report, ensure_ascii=False, indent=2))

else:
    raise ValueError("Public message removed for release.")
